In [2]:
import os
import sys
import time
import shutil

import numpy as np
from astropy.table import Table

import umap

In [11]:
#catalog name
catalog_name = "Farmer"

# #color bands
color_bands   = 'ugrizyJH'
color_columns = ['u_g', 'g_r', 'r_i', 'i_z', 'z_y', 'y_J', 'J_H']

# color_bands = ''
# color_columns = []

#magnitudes
normalize_magnitudes = False
magnitudes = ''
magnitude_columns = []
# magnitudes           = 'ugrizyJH'
# magnitude_columns    = ['CFHT_u_MAG', 'HSC_g_MAG', 'HSC_r_MAG', 'HSC_i_MAG', 'HSC_z_MAG', 'HSC_y_MAG', 'UVISTA_J_MAG', 'UVISTA_H_MAG']

#params to optimize
# list_input_distance_metric = ['euclidean', 'manhattan', 'chebyshev', 'minkowski']
list_input_distance_metric = ['manhattan']
list_min_dist              = [0.0]
list_n_neighbors           = [80]
list_n_components          = [3]

#directory to save results
save_path = '/Users/leo/Projects/umap_nz_cal/UMAPs/magcut_i_22/23feb26'

In [3]:
def normalize_magnitude_func(magnitudes, target_std = 0.5):

    bins = np.linspace(np.min(magnitudes), np.max(magnitudes), 100)
    heights, _ = np.histogram(magnitudes, bins = bins)

    mode = bins[np.argmax(heights)]
    # median = np.median(magnitudes)
    mean = np.mean(magnitudes)

    return (magnitudes - mean)/(np.abs(mode - mean)/target_std)

In [4]:
COSMOS2020_Farmer_final_dropouts = Table.read('/Users/leo/Projects/umap_nz_cal/data/COSMOS2020_Farmer_dropouts_colors_UMAP.fits', )

In [12]:
if len(color_columns) > 0:

    input_colors = COSMOS2020_Farmer_final_dropouts[color_columns]
    input_colors = np.asarray(input_colors.to_pandas(), dtype = float)

else: input_colors = None

if len(magnitude_columns) > 0:
    input_magnitudes = COSMOS2020_Farmer_final_dropouts[magnitude_columns]
    input_magnitudes = np.asarray(input_magnitudes.to_pandas(), dtype = float)
    if normalize_magnitudes:
        for i in range(np.shape(input_magnitudes)[1]):
            input_magnitudes[:, i] = normalize_magnitude_func(input_magnitudes[:, i])

else: input_magnitudes = None

input_data = np.hstack([table for table in [input_colors, input_magnitudes] if np.all(table) != None])

In [13]:
input_data = input_data[COSMOS2020_Farmer_final_dropouts['HSC_i_MAG'] < 22]

In [14]:
f = open(f'{save_path}/output_log.txt', 'w')
original = sys.stdout

class Tee:
    def write(self, data):
        original.write(data)
        f.write(data)
        f.flush()
    def flush(self):
        pass

sys.stdout = Tee()

print(f'Building UMAPs. Expect {len(list_input_distance_metric)*len(list_min_dist)*len(list_min_dist)*len(list_n_neighbors)} instances')
start_time = time.time()
print(f'Save path: {save_path}')
print(f'{"No.":<6}{"Status":<16}{"min_dist":<16}{"n_neighbors":<16}{"n_components":<16}{"instance name":<16}')
instance_number = 0
for input_distance_metric in list_input_distance_metric:
    for n_components in list_n_components:
        for min_dist in list_min_dist:
            for n_neighbors in list_n_neighbors:

                instance_number += 1
                
                cols = f'_cols{color_bands}' * bool(color_bands)
                mags = (f'_mags{magnitudes}' + '_norm' * normalize_magnitudes + '_nonorm' * (not normalize_magnitudes)) * bool(magnitudes)
                instance_name = f"UMAP_Farmer{cols}{mags}_{input_distance_metric}_mindist{str(min_dist).replace('.', 'p')}_neighb{n_neighbors}_{n_components}d"

                # print(f'{instance_number:<6}{"Building":<16}{min_dist:<16}{n_neighbors:<16}{n_components:<16}{instance_name}', end = '\r')
                try:
                    reducer = umap.UMAP(n_components = n_components, n_neighbors = n_neighbors, min_dist = min_dist, metric = input_distance_metric)
                    embedding = reducer.fit_transform(input_data)

                    np.save(f'{save_path}/{instance_name}',
                            embedding,
                            allow_pickle = True)
                
                except Exception as error:
                    traceback = error.__traceback__

                if os.path.exists(f'{save_path}/{instance_name}.npy'):
                    print(f'{instance_number:<6}{"Complete":<16}{min_dist:<16}{n_neighbors:<16}{n_components:<16}{instance_name}')

                else:
                    raise Exception('UMAP not saved to npy file')

print(f'Built {instance_number} UMAPs in {((time.time() - start_time)/60):.2f} minutes.')

sys.stdout = original
f.close()

Building UMAPs. Expect 1 instances
Save path: /Users/leo/Projects/umap_nz_cal/UMAPs/magcut_i_22/23feb26
No.   Status          min_dist        n_neighbors     n_components    instance name   
1     Complete        0.0             80              3               UMAP_Farmer_colsugrizyJH_manhattan_mindist0p0_neighb80_3d
Built 1 UMAPs in 0.07 minutes.


In [15]:
shutil.copy('/Users/leo/Projects/umap_nz_cal/code/UMAP/generate_umap.ipynb', f'{save_path}/generate_umap.ipynb')

'/Users/leo/Projects/umap_nz_cal/UMAPs/magcut_i_22/23feb26/generate_umap.ipynb'